<a href="https://colab.research.google.com/github/1Bur1/Tuwaiq-classes---advanced-AI-python---final-project/blob/main/02_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Feature Engineering

This notebook transforms the cleaned housing data to make it more useful for machine learning.  
Steps: encode categories, scale numbers, create new features, and remove redundant ones.

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
def clean_data(filepath):


    # Step 1 & 2: Load & check shape
    df = pd.read_csv(filepath)
    print(f"Loaded data: {df.shape[0]} rows, {df.shape[1]} columns")




    #Step 3: Fix data types 
    df['MS SubClass'] = df['MS SubClass'].astype('category')

    for col in ['Bsmt Full Bath', 'Bsmt Half Bath', 'Garage Cars']:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    for col in ['Year Built', 'Year Remod/Add', 'Yr Sold']:
        df[col] = pd.to_datetime(df[col], format='%Y', errors='coerce')

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')




    #Step 4: Handle missing values
    # Drop columns with more than 50% missing
    cols_to_drop = ['Pool QC', 'Misc Feature', 'Alley', 'Fence']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])


    # Fill categorical with mode
    for col in df.select_dtypes(include='category').columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0])


    # Fill numeric with median
    for col in df.select_dtypes(include=['int64', 'float64', 'Int64']).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())


    # Fill datetime with mode
    for col in df.select_dtypes(include='datetime64[ns]').columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0])




    #Step 5: Remove duplicates
    before = df.shape[0]
    df = df.drop_duplicates()
    print(f"Removed {before - df.shape[0]} duplicate rows")





    #Step 6: Cap outliers at 99th percentile 
    upper_99 = df['SalePrice'].quantile(0.99)
    df['SalePrice'] = df['SalePrice'].clip(upper=upper_99)

    print("Cleaning complete!")
    return df


df_clean = clean_data("AmesHousing.csv")
df_clean.head()

Loaded data: 2930 rows, 82 columns
Removed 0 duplicate rows
Cleaning complete!


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Lot Shape,Land Contour,Utilities,...,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,IR1,Lvl,AllPub,...,0,0,0,0,0,5,2010-01-01,WD,Normal,215000.0
1,2,526350040,20,RH,80.0,11622,Pave,Reg,Lvl,AllPub,...,0,0,120,0,0,6,2010-01-01,WD,Normal,105000.0
2,3,526351010,20,RL,81.0,14267,Pave,IR1,Lvl,AllPub,...,0,0,0,0,12500,6,2010-01-01,WD,Normal,172000.0
3,4,526353030,20,RL,93.0,11160,Pave,Reg,Lvl,AllPub,...,0,0,0,0,0,4,2010-01-01,WD,Normal,244000.0
4,5,527105010,60,RL,74.0,13830,Pave,IR1,Lvl,AllPub,...,0,0,0,0,0,3,2010-01-01,WD,Normal,189900.0


---
## Task 1 — One-Hot Encode Categorical Columns
Categorical columns like `Neighborhood` and `House Style` have no natural order.
We use `pd.get_dummies()` to convert each category into its own 0/1 column so the model can understand them.

In [57]:
#task1
#• One-hot encode at least 2 categorical columns using pd.get_dummies()


# One-hot encoding for categorical columns
df_clean = pd.get_dummies(df_clean, columns=["Neighborhood", "House Style"], prefix=["Neighborhood", "HouseStyle"])


# One-hot encoding was used for Neighborhood and House Style
# because these categorical variables have no natural order,
# and converting them into binary columns makes them usable for machine learning models.


---
## Task 2 — Ordinal Encode an Ordered Column
`Kitchen Qual` has a clear order: Poor to Fair to Typical to Good to Excellent.
We map it to numbers (0-4) so the model knows that Excellent > Good > Typical, etc.

In [58]:
#task2
#• Ordinal encode 1 ordered column (e.g., quality: low → high)


qual_order = ["Po", "Fa", "TA", "Gd", "Ex"]

df_clean["Kitchen Qual Ord"] = pd.Categorical(df_clean["Kitchen Qual"],categories=qual_order,ordered=True).codes



print(df_clean["Kitchen Qual Ord"])


#"The column 'Kitchen Qual' was selected because it represents an ordered quality scale (from poor to excellent), making ordinal encoding appropriate to preserve this natural ranking and improve model interpretability."


0       2
1       2
2       3
3       4
4       2
       ..
2925    2
2926    2
2927    2
2928    2
2929    2
Name: Kitchen Qual Ord, Length: 2930, dtype: int8


---
## Task 4 — Create Domain Features (Ratios)
We engineer 2 new features that combine existing columns in a meaningful way.
Safe division (`+ 1e-6`) is used to avoid dividing by zero.

In [59]:
#task4
#Create 2 domain features: a meaningful ratio (e.g., price_per_sqft) and one more of your choice.
#Use safe division to avoid dividing by zero



#price per square foot
df_clean["price_per_sqft"] = df_clean["SalePrice"] / (df_clean["Gr Liv Area"] + 1e-6)

# Feature 2: rooms per area (interaction / ratio)
df_clean["rooms_per_area"] = df_clean["TotRms AbvGrd"] / (df_clean["Gr Liv Area"] + 1e-6)


"""
1. price_per_sqft
"This feature was created to measure property value relative to size, providing a more comparable metric across houses."

2. rooms_per_area
"This feature captures how efficiently space is used, which may reflect layout quality and housing density."

"""


'\n1. price_per_sqft\n"This feature was created to measure property value relative to size, providing a more comparable metric across houses."\n\n2. rooms_per_area\n"This feature captures how efficiently space is used, which may reflect layout quality and housing density."\n\n'

---
## Task 5 — Create an Interaction Feature
An interaction feature multiplies two columns that are related.
`Overall Qual x Gr Liv Area` captures the idea that a big high-quality house is worth much more than a big low-quality one.

In [60]:
#task 5
#• Create 1 interaction feature: multiply two columns that logically go together (e.g., quality × area)

# Interaction feature: quality × area
df_clean["qual_area_interaction"] = df_clean["Overall Qual"] * df_clean["Gr Liv Area"]

#"This interaction feature combines quality and size, which may better capture overall property value than each feature individually."


---
## Task 6 — Log-Transform a Skewed Column
`SalePrice` is heavily skewed — a few very expensive houses pull the distribution to the right.
`np.log1p()` compresses those large values and makes the distribution more symmetric, which helps most ML models.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Task 6: Log-transform 1 skewed column using np.log1p() — show histogram before and after

# Show before and after side by side in one figure
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before (raw)
axes[0].hist(df_clean["SalePrice"], bins=40, color="steelblue", edgecolor="black")
axes[0].set_title("SalePrice — Before (Raw)")
axes[0].set_xlabel("Price ($)")
axes[0].set_ylabel("Number of Houses")

# Apply log transformation
df_clean["SalePrice_log"] = np.log1p(df_clean["SalePrice"])

# After (log)
axes[1].hist(df_clean["SalePrice_log"], bins=40, color="salmon", edgecolor="black")
axes[1].set_title("SalePrice — After log1p")
axes[1].set_xlabel("log(Price + 1)")
axes[1].set_ylabel("Number of Houses")

plt.suptitle("Effect of Log Transformation on SalePrice", fontsize=14)
plt.tight_layout()
plt.show()

# "Log transformation was applied to reduce skewness and make the distribution more
# symmetric, which helps many machine learning models learn better."

---
## Task 7 — Bin a Column into Groups
Instead of using the exact year a house was built, we group houses into 3 eras: **Old**, **Mid**, and **New**.
This makes the feature simpler and more meaningful for a model to use.

In [ ]:
# Task 7: Bin 1 column into meaningful groups (e.g., age groups: Old, Mid, New)

# Bin Year Built into 3 construction eras
# 1800 is used as the lower bound since the oldest house in the dataset is from 1872
bins = [1800, 1950, 2000, 2030]
labels = ["Old", "Mid", "New"]

df_clean["YearBuilt_bin"] = pd.cut(df_clean["Year Built"].dt.year, bins=bins, labels=labels)

# Show the count in each group
print(df_clean["YearBuilt_bin"].value_counts())

# "Binning was applied to group houses by construction era (Old: before 1950, Mid: 1950-2000, New: after 2000),
# making the feature more interpretable and capturing differences between old and modern properties."

---
## Task 8 — Remove Redundant Features
If two features are almost perfectly correlated (r > 0.95), they carry the same information.
We find those pairs and drop one to reduce noise and multicollinearity.

In [67]:
#task8
#• Remove redundant features: find highly correlated pairs (r > 0.95) and drop one





# Select only numerical columns
num_df = df_clean.select_dtypes(include=['int64', 'float64', 'Int64'])





# Exclude task-created columns you want to keep
exclude_cols = ["SalePrice", "SalePrice_capped", "SalePrice_log"]
num_df = num_df.drop(columns=[col for col in exclude_cols if col in num_df.columns])





# Compute absolute correlation matrix
corr_matrix = num_df.corr().abs()





# Keep only upper triangle
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))





# Find columns with correlation > 0.95
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]





print("Columns to drop contain high correlation:", to_drop)




# Drop redundant columns
df_clean = df_clean.drop(columns=to_drop)



#note:no highly correlated numeric feature pairs above 0.95 were found
# after excluding target-related derived columns, so no redundant feature was removed.

# "Highly correlated features were removed to reduce redundancy and multicollinearity, improving model stability and performance."

Columns to drop contain high correlation: []


---
## Task 3 — Scale Numerical Columns
Large columns like `Lot Area` can dominate a model just because of their big numbers.
`StandardScaler` rescales them so every feature has **mean = 0** and **std = 1**, putting them on equal footing.

In [ ]:
# Task 3: Scale at least 2 numerical columns using StandardScaler

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df_clean[["Gr Liv Area_scaled", "Lot Area_scaled"]] = scaler.fit_transform(
    df_clean[["Gr Liv Area", "Lot Area"]]
)

# Verify the scaling worked: mean should be ~0, std should be ~1
print("Gr Liv Area_scaled — mean:", round(df_clean["Gr Liv Area_scaled"].mean(), 4),
      "| std:", round(df_clean["Gr Liv Area_scaled"].std(), 4))
print("Lot Area_scaled    — mean:", round(df_clean["Lot Area_scaled"].mean(), 4),
      "| std:", round(df_clean["Lot Area_scaled"].std(), 4))

# "StandardScaler was applied so both features have mean 0 and standard deviation 1,
# preventing large-scale features from dominating the model."